# Exercise 6: Query Phrasing Sensitivity

The same underlying question phrased 5+ different ways — does retrieval quality change?

**Goal:** Understand how phrasing affects vector similarity search.


In [1]:
# Install required packages
try:
    ip = get_ipython()
    ip.run_line_magic('pip', 'install -q torch transformers sentence-transformers faiss-cpu pymupdf accelerate')
except NameError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'torch', 'transformers', 'sentence-transformers', 'faiss-cpu', 'pymupdf', 'accelerate'])


In [2]:
import os, sys
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import torch
from typing import List, Tuple
from pathlib import Path

def detect_environment():
    try:
        import google.colab
        return 'colab'
    except ImportError:
        return 'local'

def get_device():
    if torch.cuda.is_available():
        device, dtype = 'cuda', torch.float16
        print(f"✓ CUDA: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")
    elif torch.backends.mps.is_available():
        device, dtype = 'mps', torch.float32
        print("✓ Apple Silicon MPS")
    else:
        device, dtype = 'cpu', torch.float32
        print("⚠ CPU only")
    return device, dtype

ENVIRONMENT = detect_environment()
DEVICE, DTYPE = get_device()
print(f"Environment: {ENVIRONMENT.upper()} | Device: {DEVICE} | Dtype: {DTYPE}")


✓ CUDA: Tesla T4 (15.6 GB)
Environment: COLAB | Device: cuda | Dtype: torch.float16


In [3]:
from pathlib import Path

CORPUS_SUBFOLDER = "ModelTService"  # <- change if needed
DRIVE_BASE = '/content/drive/MyDrive/Colab_Projects/Week05-RAG/Corpora'

if ENVIRONMENT == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    DOC_FOLDER = f"{DRIVE_BASE}/{CORPUS_SUBFOLDER}"
else:
    DOC_FOLDER = str(Path("Corpora") / CORPUS_SUBFOLDER)

print(f"DOC_FOLDER = {DOC_FOLDER}")
assert Path(DOC_FOLDER).exists(), f"Folder not found: {DOC_FOLDER}"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DOC_FOLDER = /content/drive/MyDrive/Colab_Projects/Week05-RAG/Corpora/ModelTService


In [4]:
import fitz

def load_text_file(fp):
    with open(fp, 'r', encoding='utf-8', errors='ignore') as f:
        return f.read()

def load_pdf_file(fp):
    doc = fitz.open(fp)
    parts = []
    for i, page in enumerate(doc):
        t = page.get_text()
        if t.strip():
            parts.append(f"\n[Page {i+1}]\n{t}")
    doc.close()
    return "\n".join(parts)

def load_documents(folder):
    docs = []
    for fp in Path(folder).rglob("*"):
        if not fp.is_file(): continue
        if fp.suffix.lower() not in ('.pdf', '.txt', '.md'): continue
        try:
            content = load_pdf_file(str(fp)) if fp.suffix.lower() == '.pdf' else load_text_file(str(fp))
            if content.strip():
                docs.append((fp.name, content))
                print(f"✓ {fp.name} ({len(content):,} chars)")
        except Exception as e:
            print(f"✗ {fp.name}: {e}")
    return docs

documents = load_documents(DOC_FOLDER)
print(f"\nLoaded {len(documents)} documents")


✓ Ford-Model-T-Man-1919.txt (95,574 chars)
✓ ModelT-01-10.txt (18,676 chars)
✓ ModelT-11-20.txt (19,009 chars)
✓ ModelT-21-30.txt (17,050 chars)
✓ ModelT-31-40.txt (12,194 chars)
✓ ModelT-41-50.txt (14,264 chars)
✓ ModelT-51-60.txt (14,168 chars)
✓ ModelT-61-62.txt (201 chars)
✓ Ford-Model-T-Man-1919-ocr.pdf (95,517 chars)
✓ ModelT-01-10-ocr.pdf (18,658 chars)
✓ ModelT-11-20-ocr.pdf (19,003 chars)
✓ ModelT-21-30-ocr.pdf (17,025 chars)
✓ ModelT-31-40-ocr.pdf (12,201 chars)
✓ ModelT-41-50-ocr.pdf (14,270 chars)
✓ ModelT-51-60-ocr.pdf (14,107 chars)
✓ ModelT-61-62-ocr.pdf (204 chars)

Loaded 16 documents


In [5]:
from dataclasses import dataclass

@dataclass
class Chunk:
    text: str
    source_file: str
    chunk_index: int
    start_char: int
    end_char: int

def chunk_text(text, source_file, chunk_size=512, chunk_overlap=128):
    chunks, start, idx = [], 0, 0
    while start < len(text):
        end = start + chunk_size
        if end < len(text):
            pb = text.rfind('\n\n', start + chunk_size // 2, end)
            if pb != -1:
                end = pb + 2
            else:
                sb = text.rfind('. ', start + chunk_size // 2, end)
                if sb != -1:
                    end = sb + 2
        s = text[start:end].strip()
        if s:
            chunks.append(Chunk(s, source_file, idx, start, end))
            idx += 1
        prev_start = start
        start = end - chunk_overlap
        if chunks and start <= chunks[-1].start_char:
            start = end
    return chunks

# Default chunking parameters (override per exercise)
CHUNK_SIZE    = 512
CHUNK_OVERLAP = 128

all_chunks = []
for fname, content in documents:
    dc = chunk_text(content, fname, CHUNK_SIZE, CHUNK_OVERLAP)
    all_chunks.extend(dc)
    print(f"  {fname}: {len(dc)} chunks")
print(f"\nTotal chunks: {len(all_chunks)}")


  Ford-Model-T-Man-1919.txt: 326 chunks
  ModelT-01-10.txt: 64 chunks
  ModelT-11-20.txt: 66 chunks
  ModelT-21-30.txt: 56 chunks
  ModelT-31-40.txt: 44 chunks
  ModelT-41-50.txt: 51 chunks
  ModelT-51-60.txt: 46 chunks
  ModelT-61-62.txt: 1 chunks
  Ford-Model-T-Man-1919-ocr.pdf: 316 chunks
  ModelT-01-10-ocr.pdf: 63 chunks
  ModelT-11-20-ocr.pdf: 61 chunks
  ModelT-21-30-ocr.pdf: 56 chunks
  ModelT-31-40-ocr.pdf: 44 chunks
  ModelT-41-50-ocr.pdf: 47 chunks
  ModelT-51-60-ocr.pdf: 44 chunks
  ModelT-61-62-ocr.pdf: 1 chunks

Total chunks: 1286


In [6]:
from sentence_transformers import SentenceTransformer
import numpy as np

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
print(f"Loading: {EMBEDDING_MODEL} on {DEVICE}")
embed_model = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)
EMBEDDING_DIM = embed_model.get_sentence_embedding_dimension()
print(f"Embedding dim: {EMBEDDING_DIM}")


Loading: sentence-transformers/all-MiniLM-L6-v2 on cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding dim: 384


In [7]:
import faiss, pickle
from pathlib import Path

# Cache config — CACHE_KEY encodes corpus + chunk params
# Change it if you change CORPUS_SUBFOLDER, CHUNK_SIZE, or CHUNK_OVERLAP
CACHE_DIR   = Path("/content/drive/MyDrive/Colab_Projects/Week05-RAG/cache")
CACHE_KEY   = "modelTService_512_128"
CHUNKS_FILE = CACHE_DIR / f"{CACHE_KEY}.chunks.pkl"
EMBEDS_FILE = CACHE_DIR / f"{CACHE_KEY}.embeddings.npy"
INDEX_FILE  = CACHE_DIR / f"{CACHE_KEY}.faiss"
CACHE_DIR.mkdir(exist_ok=True)

def save_cache():
    with open(CHUNKS_FILE, "wb") as f: pickle.dump(all_chunks, f)
    np.save(str(EMBEDS_FILE), chunk_embeddings)
    faiss.write_index(index, str(INDEX_FILE))
    print(f"✓ Cache saved → {CACHE_DIR}/{CACHE_KEY}.*")

def load_cache():
    global all_chunks, chunk_embeddings, index
    with open(CHUNKS_FILE, "rb") as f: all_chunks = pickle.load(f)
    chunk_embeddings = np.load(str(EMBEDS_FILE))
    index = faiss.read_index(str(INDEX_FILE))
    print(f"✓ Cache loaded: {len(all_chunks)} chunks, {index.ntotal} vectors")

if CHUNKS_FILE.exists() and EMBEDS_FILE.exists() and INDEX_FILE.exists():
    load_cache()
else:
    print("No cache found — embedding chunks (first run only, will be cached after)...")
    chunk_embeddings = embed_model.encode(
        [c.text for c in all_chunks], show_progress_bar=True
    ).astype("float32")
    index = faiss.IndexFlatIP(EMBEDDING_DIM)
    faiss.normalize_L2(chunk_embeddings)
    index.add(chunk_embeddings)
    print(f"✓ Index built: {index.ntotal} vectors")
    save_cache()

def rebuild_pipeline(chunk_size=512, chunk_overlap=128):
    """Re-chunk, re-embed, rebuild index in-memory (for Ex 7 & 8 experiments)."""
    global all_chunks, chunk_embeddings, index
    all_chunks = []
    for fname, content in documents:
        all_chunks.extend(chunk_text(content, fname, chunk_size, chunk_overlap))
    chunk_embeddings = embed_model.encode(
        [c.text for c in all_chunks], show_progress_bar=True
    ).astype("float32")
    faiss.normalize_L2(chunk_embeddings)
    index = faiss.IndexFlatIP(EMBEDDING_DIM)
    index.add(chunk_embeddings)
    print(f"Rebuilt: {len(all_chunks)} chunks | size={chunk_size}, overlap={chunk_overlap}")

def retrieve(query, top_k=5):
    qe = embed_model.encode([query]).astype("float32")
    faiss.normalize_L2(qe)
    scores, idxs = index.search(qe, top_k)
    return [(all_chunks[i], float(s)) for s, i in zip(scores[0], idxs[0]) if i != -1]


✓ Cache loaded: 1286 chunks, 1286 vectors


In [8]:
from transformers import AutoModelForCausalLM, AutoTokenizer

LLM_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"Loading LLM: {LLM_MODEL} on {DEVICE}...")
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)

if DEVICE == 'cuda':
    model = AutoModelForCausalLM.from_pretrained(LLM_MODEL, device_map="auto",
                torch_dtype=DTYPE, trust_remote_code=True)
elif DEVICE == 'mps':
    model = AutoModelForCausalLM.from_pretrained(LLM_MODEL, torch_dtype=DTYPE,
                trust_remote_code=True).to(DEVICE)
else:
    model = AutoModelForCausalLM.from_pretrained(LLM_MODEL, torch_dtype=DTYPE,
                trust_remote_code=True)
print("LLM loaded ✓")


Loading LLM: Qwen/Qwen2.5-1.5B-Instruct on cuda...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

LLM loaded ✓


In [11]:
PROMPT_TEMPLATE = """You are a helpful assistant that answers questions based on the provided context.

CONTEXT:
{context}

QUESTION: {question}

INSTRUCTIONS:
- Answer ONLY based on the information in the context above.
- If the context doesn't contain the answer, say so explicitly.
- Quote relevant passages to support your answer.
- Be concise and direct.

ANSWER:"""

def generate_response(prompt, max_new_tokens=512, temperature=0.3):
    inputs = tokenizer(prompt, return_tensors="pt")
    if DEVICE == 'cuda':
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
    else:
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             temperature=temperature,
                             do_sample=(temperature > 0),
                             pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

def direct_query(question, max_new_tokens=512):
    prompt = f"Answer this question:\n{question}\n\nAnswer:"
    return generate_response(prompt, max_new_tokens)

def rag_query(question, top_k=5, show_context=False, prompt_template=None):
    results = retrieve(question, top_k)
    ctx_parts = [f"[Source: {c.source_file}, Score: {s:.3f}]\n{c.text}" for c, s in results]
    context = "\n\n---\n\n".join(ctx_parts)
    if show_context:
        print("=== RETRIEVED CONTEXT ==="); print(context); print("=" * 40)
    tmpl = prompt_template if prompt_template else PROMPT_TEMPLATE
    return generate_response(tmpl.format(context=context, question=question))


In [12]:
import numpy as np

PHRASINGS = {
    "carburetor_adjustment": [
        "What is the recommended maintenance schedule for the carburetor?",
        "How often should I service the carburetor?",
        "carburetor maintenance intervals",
        "When do I need to check the carburetor?",
        "Preventive maintenance requirements for carburetor",
        "How do I adjust the carburetor on a Model T?",
    ],
    "spark_plug": [
        "What spark plug gap specification does the manual list?",
        "spark plug gap setting",
        "How wide should the spark plug gap be?",
        "electrode distance for Model T spark plugs",
        "ignition plug clearance",
    ],
}

def jaccard(set_a, set_b):
    """Chunk text Jaccard overlap between two result sets."""
    a = set(t.text for t, _ in set_a)
    b = set(t.text for t, _ in set_b)
    if not a | b:
        return 0.0
    return len(a & b) / len(a | b)

for topic, phrasings in PHRASINGS.items():
    print(f"\n{'='*70}")
    print(f"TOPIC: {topic}")
    print(f"{'='*70}")

    all_results = []
    for phrase in phrasings:
        results = retrieve(phrase, top_k=5)
        all_results.append(results)
        scores = [s for _, s in results]
        sources = [c.source_file + f"[{c.chunk_index}]" for c, _ in results]
        print(f"\n  Phrasing: '{phrase}'")
        print(f"  Scores:  {[f'{s:.3f}' for s in scores]}")
        print(f"  Sources: {sources[:3]}")
        print(f"  Top chunk: {results[0][0].text[:120]}...")

    # Pairwise Jaccard overlap matrix
    print(f"\n  Jaccard overlap (pairwise, source texts):")
    for i in range(len(phrasings)):
        row = []
        for j in range(len(phrasings)):
            row.append(f"{jaccard(all_results[i], all_results[j]):.2f}")
        print(f"  {phrasings[i][:30]:30} {row}")



TOPIC: carburetor_adjustment

  Phrasing: 'What is the recommended maintenance schedule for the carburetor?'
  Scores:  ['0.415', '0.415', '0.410', '0.410', '0.381']
  Sources: ['ModelT-21-30-ocr.pdf[18]', 'Ford-Model-T-Man-1919-ocr.pdf[142]', 'ModelT-01-10.txt[34]']
  Top chunk: mplish the purpose, the carburetor should be drained.
If Engine runs too fast or chokes
with throttle retarded—what? Ans...

  Phrasing: 'How often should I service the carburetor?'
  Scores:  ['0.497', '0.497', '0.462', '0.462', '0.449']
  Sources: ['ModelT-01-10.txt[34]', 'Ford-Model-T-Man-1919.txt[34]', 'ModelT-21-30-ocr.pdf[18]']
  Top chunk: ll of a very rich gasoline vapor. This should not.
Gone unless the car is going to stand over night or long enough to co...

  Phrasing: 'carburetor maintenance intervals'
  Scores:  ['0.501', '0.501', '0.491', '0.491', '0.466']
  Sources: ['ModelT-01-10.txt[34]', 'Ford-Model-T-Man-1919.txt[34]', 'ModelT-21-30-ocr.pdf[18]']
  Top chunk: ll of a very rich gasoline vap

## Analysis

**Carburetor topic — retrieval scores and Jaccard overlaps:**

| Phrasing | Top Score | Cluster |
|----------|-----------|---------|
| "What is the recommended maintenance schedule for the carburetor?" | 0.415 | Maintenance |
| "How often should I service the carburetor?" | 0.497 | Maintenance |
| "carburetor maintenance intervals" | 0.501 | Maintenance |
| "When do I need to check the carburetor?" | 0.569 | Maintenance |
| "Preventive maintenance requirements for carburetor" | 0.515 | Maintenance |
| **"How do I adjust the carburetor on a Model T?"** | **0.640** | **Adjustment** |

The action-oriented phrasing ("adjust") scored 0.640 — 13% higher than the best maintenance phrasing — and retrieved **entirely different chunks** (Jaccard = 0.00 with all five maintenance phrasings). The five maintenance phrasings clustered among themselves (Jaccard ≈ 0.50), all retrieving draining/troubleshooting chunks rather than adjustment procedure chunks.

**Spark plug topic — retrieval scores:**

| Phrasing | Top Score | Cluster |
|----------|-----------|---------|
| "What spark plug gap specification does the manual list?" | 0.601 | Dirty-plugs / spec |
| "spark plug gap setting" | 0.598 | Gap value chunk |
| "How wide should the spark plug gap be?" | 0.537 | Dirty-plugs / spec |
| "electrode distance for Model T spark plugs" | 0.490 | Gap value chunk (partial) |
| **"ignition plug clearance"** | 0.526 | **Completely different cluster** |

"Ignition plug clearance" retrieved replacement/porcelain chunks (Jaccard 0.00 with question-form phrasings), showing that domain terminology shifts cause dramatic retrieval divergence even for the same underlying question.

**Answers to analysis questions:**

- **Which phrasings retrieved the best chunks?** Action-specific phrasings ("How do I adjust…") consistently yield higher scores and more relevant chunks than generic maintenance framing ("How often should I service…"). For spark plug, the natural-language question forms ("gap specification", "gap setting") outperformed technical synonyms ("electrode distance", "ignition plug clearance").
- **Do keyword-style queries work better or worse?** Comparably to natural-language questions *within the same semantic cluster* ("carburetor maintenance intervals" ≈ "How often should I service"), but they don't escape the cluster. Keyword style is neither reliably better nor worse — what matters is whether the keywords land in the right embedding region.
- **Query rewriting implications:** The 0.00 Jaccard between "adjust" and "maintenance" phrasings is the key finding. A single phrasing can miss 100% of the most relevant chunks. Practical strategies: (1) generate 2-3 phrasings with different action verbs and union the retrieved sets; (2) use HyDE (hypothetical document embeddings) to anchor queries closer to the manual's phrasing; (3) prepend the domain ("Model T Ford manual: …") to reduce semantic drift.

In [ ]:
import pickle

def save_index(path):
    faiss.write_index(index, f"{path}.faiss")
    with open(f"{path}.chunks", 'wb') as f:
        pickle.dump(all_chunks, f)
    print(f"✓ Saved {path}.faiss + .chunks")

save_index("rag_index")
